# Virtual-pipeline — VTuber Builder

이 노트북은 **사용자 UI 중심**으로 동작합니다.

1. 첫 번째 **Setup** 셀만 실행합니다.
2. 설치가 끝나면 아래 **VTuber Builder UI** 셀을 실행합니다.
3. 화면에서 **Avatar 만들기 / Accessory 붙이기**를 선택하고 파일을 업로드한 뒤 **실행**을 누릅니다.
4. 성공하면 **VRM 다운로드** 버튼이 활성화됩니다.

### GitHub 인증
- 저장소가 **public이면 GitHub 토큰을 전혀 요구하지 않습니다.**
- 저장소가 private인 경우에만 익명 clone/fetch가 실패한 뒤 토큰을 요청합니다.
- private 저장소를 계속 쓸 때는 Colab Secret에 `GITHUB_TOKEN`을 한 번 등록해두면 입력창도 뜨지 않습니다.

Setup은 항상 최신 `origin/main`으로 동기화한 뒤 그 checkout을 설치합니다.


## 1. Setup — 최신 main + dependency 설치


In [ ]:
import getpass, os, pathlib, shutil, subprocess, sys

REPO_URL = 'https://github.com/jujumelona/Virtual-pipeline.git'
REPO_DIR = pathlib.Path('/content/Virtual-pipeline')
TRIPOSR_DIR = pathlib.Path('/content/third_party/TripoSR')
TRIPOSR_COMMIT = '107cefdc244c39106fa830359024f6a2f1c78871'

if sys.version_info < (3, 12):
    raise RuntimeError(f'Python 3.12+ required, current={sys.version}')

def run(cmd, *, env=None, quiet=False):
    if not quiet:
        safe = ['***' if 'token' in str(x).lower() else str(x) for x in cmd]
        print('+', ' '.join(safe))
    subprocess.run(cmd, env=env, check=True)

def sync_repo(env=None):
    if (REPO_DIR / '.git').is_dir():
        run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL], env=env)
        run(['git', '-C', str(REPO_DIR), 'fetch', '--prune', 'origin', 'main'], env=env)
    else:
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        run(['git', 'clone', '--branch', 'main', '--single-branch', REPO_URL, str(REPO_DIR)], env=env)
    run(['git', '-C', str(REPO_DIR), 'checkout', '-B', 'main', 'origin/main'], env=env)
    run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], env=env)

print('1/4 최신 Virtual-pipeline main 동기화...')
repo_access = 'public'
try:
    sync_repo()
except subprocess.CalledProcessError:
    repo_access = 'private'
    if REPO_DIR.exists() and not (REPO_DIR / '.git').is_dir():
        shutil.rmtree(REPO_DIR)

    github_token = None
    try:
        from google.colab import userdata
        try:
            github_token = userdata.get('GITHUB_TOKEN')
        except Exception:
            github_token = None
    except Exception:
        github_token = None

    if not github_token:
        print('이 저장소는 익명 접근이 되지 않습니다. private 저장소인 경우에만 토큰이 필요합니다.')
        github_token = getpass.getpass('GitHub token (Contents: Read): ').strip()
    if not github_token:
        raise RuntimeError(
            'Private repository access requires GITHUB_TOKEN. '
            'Repository를 public으로 바꾸면 이 입력은 사라집니다.'
        )

    askpass = pathlib.Path('/tmp/virtual-pipeline-git-askpass.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n'
    )
    askpass.chmod(0o700)
    git_env = os.environ.copy()
    git_env.update({
        'GITHUB_TOKEN': github_token,
        'GIT_ASKPASS': str(askpass),
        'GIT_TERMINAL_PROMPT': '0',
    })
    try:
        sync_repo(git_env)
    finally:
        github_token = None
        git_env.pop('GITHUB_TOKEN', None)
        try:
            askpass.unlink()
        except FileNotFoundError:
            pass

head = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
origin_head = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'origin/main'], text=True
).strip()
if head != origin_head:
    raise RuntimeError(f'latest main sync failed: local={head}, origin/main={origin_head}')
print('   Virtual-pipeline:', head, f'({repo_access})')

print('2/4 pinned TripoSR 준비...')
TRIPOSR_DIR.parent.mkdir(parents=True, exist_ok=True)
if (TRIPOSR_DIR / '.git').is_dir():
    run(['git', '-C', str(TRIPOSR_DIR), 'fetch', '--prune', 'origin'])
else:
    if TRIPOSR_DIR.exists():
        shutil.rmtree(TRIPOSR_DIR)
    run(['git', 'clone', 'https://github.com/VAST-AI-Research/TripoSR.git', str(TRIPOSR_DIR)])
run(['git', '-C', str(TRIPOSR_DIR), 'checkout', '--detach', TRIPOSR_COMMIT])
actual_triposr = subprocess.check_output(
    ['git', '-C', str(TRIPOSR_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
if actual_triposr != TRIPOSR_COMMIT:
    raise RuntimeError(f'TripoSR pin mismatch: {actual_triposr}')

print('3/4 Python dependency 설치...')
run([sys.executable, '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel'])
run([sys.executable, '-m', 'pip', 'install', '-r', str(TRIPOSR_DIR / 'requirements.txt')])
run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR)])
run([sys.executable, '-m', 'pip', 'install', '-U', 'ipywidgets'])

os.environ['TRIPOSR_DIR'] = str(TRIPOSR_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

for module_name in list(sys.modules):
    if module_name == 'vtuber_pipeline' or module_name.startswith('vtuber_pipeline.'):
        del sys.modules[module_name]

print('4/4 완료')
print('✅ Setup complete')
print('   main:', head)
print('   TripoSR:', actual_triposr)
print('아래 VTuber Builder UI 셀을 실행하세요.')


## 2. VTuber Builder UI


In [ ]:
import json
import pathlib
import traceback
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

from vtuber_pipeline.avatar import build_avatar
from vtuber_pipeline.accessory import reconstruct_accessories, AccessoryPipeline

UI_ROOT = pathlib.Path('/content/vtuber_builder')
INPUT_ROOT = UI_ROOT / 'input'
OUTPUT_ROOT = UI_ROOT / 'output'
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ANCHORS = [
    'HEAD_TOP', 'FACE', 'LEFT_EAR', 'RIGHT_EAR', 'NECK', 'CHEST', 'BACK',
    'LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_HAND', 'RIGHT_HAND',
    'LEFT_FOOT', 'RIGHT_FOOT', 'HIPS',
]

state = {
    'avatar_vrm': None,
    'result_vrm': None,
}

def upload_entries(upload):
    value = upload.value
    if not value:
        return []
    if isinstance(value, dict):
        out = []
        for name, item in value.items():
            content = item.get('content', b'') if isinstance(item, dict) else item
            out.append((name, bytes(content)))
        return out
    out = []
    for item in value:
        if isinstance(item, dict):
            out.append((item.get('name', 'upload.bin'), bytes(item.get('content', b''))))
        else:
            name = getattr(item, 'name', 'upload.bin')
            content = getattr(item, 'content', b'')
            out.append((name, bytes(content)))
    return out

def save_uploads(upload, directory):
    directory = pathlib.Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    paths = []
    for name, data in upload_entries(upload):
        path = directory / pathlib.Path(name).name
        path.write_bytes(data)
        paths.append(path)
    return paths

title = widgets.HTML(
    value='<h2 style="margin:0">VTuber Builder</h2>'
          '<div style="color:#666">이미지를 올리고 모드를 선택한 뒤 실행만 누르세요.</div>'
)

mode = widgets.ToggleButtons(
    options=[('Avatar 만들기', 'avatar'), ('Accessory 붙이기', 'accessory')],
    value='avatar',
    description='모드',
    button_style='',
)

commercial_usage = widgets.Dropdown(
    options=[
        ('기업/수익 사용 허용', 'corporation'),
        ('개인 수익 사용', 'personalProfit'),
        ('개인 비영리', 'personalNonProfit'),
    ],
    value='corporation',
    description='이용 범위',
)

avatar_upload = widgets.FileUpload(
    accept='image/png,image/jpeg,image/webp',
    multiple=False,
    description='캐릭터 이미지',
)
avatar_help = widgets.HTML(
    '<b>Avatar 입력</b><br>'
    '정면에 가까운 캐릭터 이미지 1장을 올리세요. '
    '얼굴/머리/상체가 충분히 보이는 이미지가 좋습니다.'
)

use_recent_avatar = widgets.Checkbox(
    value=True,
    description='방금 만든 Avatar VRM 사용',
    indent=False,
)
base_vrm_upload = widgets.FileUpload(
    accept='.vrm',
    multiple=False,
    description='Base VRM',
)
accessory_upload = widgets.FileUpload(
    accept='image/png,image/jpeg,image/webp',
    multiple=True,
    description='악세사리 이미지',
)
physics_enabled = widgets.Checkbox(
    value=False,
    description='동적 physics 사용 (실험적)',
    indent=False,
)
anchor_box = widgets.VBox([])
anchor_selectors = {}

def refresh_anchors(*_):
    anchor_selectors.clear()
    rows = []
    for name, _data in upload_entries(accessory_upload):
        dropdown = widgets.Dropdown(
            options=ANCHORS,
            value='HEAD_TOP',
            description='Anchor',
            layout=widgets.Layout(width='310px'),
        )
        anchor_selectors[name] = dropdown
        rows.append(widgets.HBox([
            widgets.HTML(f'<div style="width:220px;overflow:hidden"><b>{name}</b></div>'),
            dropdown,
        ]))
    if not rows:
        rows = [widgets.HTML('<span style="color:#777">악세사리 이미지를 올리면 파일별 anchor가 여기에 나타납니다.</span>')]
    anchor_box.children = tuple(rows)

accessory_upload.observe(refresh_anchors, names='value')
refresh_anchors()

avatar_panel = widgets.VBox([
    avatar_help,
    avatar_upload,
    commercial_usage,
])

accessory_panel = widgets.VBox([
    widgets.HTML(
        '<b>Accessory 입력</b><br>'
        '완성된 VRM에 악세사리 이미지를 여러 장 붙일 수 있습니다. '
        '각 파일마다 anchor를 선택하세요.'
    ),
    use_recent_avatar,
    base_vrm_upload,
    accessory_upload,
    widgets.HTML('<b>파일별 Anchor</b>'),
    anchor_box,
    physics_enabled,
])
accessory_panel.layout.display = 'none'

run_button = widgets.Button(
    description='Avatar 생성 실행',
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='220px', height='40px'),
)
download_button = widgets.Button(
    description='VRM 다운로드',
    button_style='success',
    icon='download',
    disabled=True,
    layout=widgets.Layout(width='180px', height='40px'),
)
status_html = widgets.HTML('<span style="color:#777">대기 중</span>')
log_output = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ddd',
        padding='8px',
        max_height='420px',
        overflow='auto',
    )
)

def set_mode(change=None):
    is_avatar = mode.value == 'avatar'
    avatar_panel.layout.display = '' if is_avatar else 'none'
    accessory_panel.layout.display = 'none' if is_avatar else ''
    run_button.description = 'Avatar 생성 실행' if is_avatar else 'Accessory 적용 실행'
    download_button.disabled = True
    state['result_vrm'] = None
    status_html.value = '<span style="color:#777">대기 중</span>'

mode.observe(set_mode, names='value')
set_mode()

def stage_line(name, result):
    if not isinstance(result, dict):
        return
    status = result.get('status', 'unknown')
    detail = result.get('error') or result.get('warning') or ''
    mark = '✓' if status == 'complete' else ('·' if status in {'cached', 'skipped'} else '✗')
    print(f'{mark} {name}: {status}' + (f' — {detail}' if detail else ''))

def run_avatar_mode():
    paths = save_uploads(avatar_upload, INPUT_ROOT / 'avatar')
    if len(paths) != 1:
        raise ValueError('캐릭터 이미지 1장을 업로드하세요.')

    out_dir = OUTPUT_ROOT / 'avatar'
    print('Avatar 입력:', paths[0])
    print('최신 main:', globals().get('head', 'unknown'))
    result = build_avatar(
        image_path=str(paths[0]),
        output_dir=str(out_dir),
        config={
            'profile': 'commercial',
            'commercial_usage': commercial_usage.value,
        },
    )

    for name, stage in result.get('stages', {}).items():
        stage_line(name, stage)

    if result.get('status') != 'complete':
        raise RuntimeError(
            result.get('failed_reason')
            or f"Avatar build failed: {result.get('failed_stages') or result.get('status')}"
        )

    vrm = pathlib.Path(result['vrm_path'])
    if not vrm.is_file():
        raise RuntimeError(f'VRM output missing: {vrm}')

    state['avatar_vrm'] = str(vrm)
    state['result_vrm'] = str(vrm)
    return result

def resolve_base_vrm():
    if use_recent_avatar.value and state.get('avatar_vrm'):
        path = pathlib.Path(state['avatar_vrm'])
        if path.is_file():
            return path

    paths = save_uploads(base_vrm_upload, INPUT_ROOT / 'base_vrm')
    if len(paths) != 1:
        raise ValueError(
            '방금 만든 Avatar VRM이 없으면 Base VRM 파일 1개를 업로드하세요.'
        )
    return paths[0]

def run_accessory_mode():
    base_vrm = resolve_base_vrm()
    accessory_paths = save_uploads(accessory_upload, INPUT_ROOT / 'accessories')
    if not accessory_paths:
        raise ValueError('악세사리 이미지를 1장 이상 업로드하세요.')

    refresh_anchors()
    print('Base VRM:', base_vrm)
    print('Accessory images:', len(accessory_paths))

    recon_dir = OUTPUT_ROOT / 'accessory_reconstruction'
    reconstructed = reconstruct_accessories(
        [str(p) for p in accessory_paths],
        str(recon_dir),
        profile='commercial',
    )

    current_vrm = str(base_vrm)
    for index, (source_path, item) in enumerate(zip(accessory_paths, reconstructed)):
        if item.get('status') != 'complete' or not item.get('mesh'):
            raise RuntimeError(
                f'Accessory reconstruction failed ({source_path.name}): '
                f'{item.get("error") or item}'
            )

        selector = anchor_selectors.get(source_path.name)
        anchor = selector.value if selector else 'HEAD_TOP'
        print(f'[{index+1}/{len(accessory_paths)}] {source_path.name} → {anchor}')

        item_dir = OUTPUT_ROOT / f'accessory_{index:03d}'
        result = AccessoryPipeline(str(item_dir)).build(
            base_vrm=current_vrm,
            accessory_glb=item['mesh'],
            config={
                'anchor_name': anchor,
                'physics': {'enabled': bool(physics_enabled.value)},
            },
        )
        for stage_name, stage in result.get('stages', {}).items():
            stage_line(f'{source_path.name}/{stage_name}', stage)

        if result.get('status') != 'complete':
            raise RuntimeError(
                f'Accessory pipeline failed ({source_path.name}): '
                f'{result.get("failed_stages") or result.get("incomplete_stages") or result.get("status")}'
            )
        current_vrm = result['output_vrm']

    final_vrm = pathlib.Path(current_vrm)
    if not final_vrm.is_file():
        raise RuntimeError(f'Combined VRM output missing: {final_vrm}')

    state['result_vrm'] = str(final_vrm)
    return {'status': 'complete', 'vrm_path': str(final_vrm)}

def on_run(_button):
    run_button.disabled = True
    download_button.disabled = True
    state['result_vrm'] = None
    status_html.value = '<b style="color:#1a73e8">실행 중...</b>'
    with log_output:
        clear_output(wait=True)
        try:
            if mode.value == 'avatar':
                result = run_avatar_mode()
            else:
                result = run_accessory_mode()
            vrm_path = result.get('vrm_path') or state.get('result_vrm')
            print()
            print('✅ 완료:', vrm_path)
            status_html.value = f'<b style="color:#188038">완료</b> — {vrm_path}'
            download_button.disabled = False
        except Exception as exc:
            print()
            print('❌ 실패:', exc)
            traceback.print_exc()
            status_html.value = f'<b style="color:#d93025">실패</b> — {exc}'
        finally:
            run_button.disabled = False

def on_download(_button):
    path = state.get('result_vrm')
    if not path or not pathlib.Path(path).is_file():
        status_html.value = '<b style="color:#d93025">다운로드할 VRM이 없습니다.</b>'
        return
    if colab_files is None:
        print('결과 파일:', path)
        return
    colab_files.download(path)

run_button.on_click(on_run)
download_button.on_click(on_download)

display(HTML(
    '<style>'
    '.widget-label { font-weight: 600 !important; }'
    '</style>'
))
display(widgets.VBox([
    title,
    widgets.HTML('<hr>'),
    mode,
    avatar_panel,
    accessory_panel,
    widgets.HTML('<hr>'),
    widgets.HBox([run_button, download_button]),
    status_html,
    widgets.HTML('<b>진행 로그</b>'),
    log_output,
]))
